# AI AINAK — Notebook 5: Complete Multimodal Assistive Demonstration

**Flagship Presentation Notebook:** End-to-End Execution of the Complete AI AINAK Pipeline on Recorded Multimodal Sessions.

This notebook integrates all sensory branches and downstream decision layers:
1. **Sensory Ingestion:** Loads synchronized MP4 video and WAV audio from `session-multimodal-01`.
2. **Computer Vision:** Runs YOLOv8n, IoU tracking, relative depth, and looming hazard estimation.
3. **Environmental Audio:** Runs PANNs CNN14 feature extraction and downstream AudioMLP classification.
4. **Kinematic Gesture:** Runs MediaPipe Pose Tasks 33-point keypoint extraction and GestureLSTM sequence classification.
5. **Timestamp Alignment:** Synchronizes observation streams within a sliding temporal window ($\Delta t \le 1.5\,	ext{s}$).
6. **Transformer Fusion:** Executes the trained 2-layer 4-head Multimodal Transformer (`fusion.pt`).
7. **Decision Engine:** Evaluates deterministic policy rules, urgency bands, and refractory cooldown suppression.
8. **Speech Output:** Synthesizes actionable spoken alerts with interactive audio playback inside the notebook.
9. **Interactive Dashboard View:** Renders composite multi-sensor inspection views suitable for faculty evaluators.


In [ ]:
# Step 1: Install Dependencies
!pip install -q ultralytics panns-inference mediapipe soundfile librosa torch numpy matplotlib gTTS


In [ ]:
# Environment Detection & Path Resolution
import os, sys, hashlib
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    print("Running in Google Colab environment.")
    from google.colab import drive
    # Mount Google Drive if available
    try:
        drive.mount("/content/drive")
    except Exception as e:
        print(f"Drive mount notice: {e}")
    
    # Locate project root
    drive_candidates = [
        Path("/content/drive/MyDrive/AINAK_Colab"),
        Path("/content/drive/MyDrive/AINAK"),
        Path("/content/AINAK"),
        Path("/content"),
    ]
    PROJECT_ROOT = next((p for p in drive_candidates if (p / "artifacts").is_dir() or (p / "checkpoints").is_dir()), Path.cwd())
else:
    print("Running in local environment.")
    PROJECT_ROOT = Path.cwd().resolve()
    while not (PROJECT_ROOT / "artifacts").is_dir() and PROJECT_ROOT.parent != PROJECT_ROOT:
        PROJECT_ROOT = PROJECT_ROOT.parent

print(f"Project root resolved to: {PROJECT_ROOT}")

# Helper to find checkpoint across standard folders
def find_checkpoint(filename: str, subfolder: str = "") -> Path:
    candidates = [
        PROJECT_ROOT / "artifacts" / subfolder / filename,
        PROJECT_ROOT / "artifacts" / filename,
        PROJECT_ROOT / "checkpoints" / subfolder / filename,
        PROJECT_ROOT / "checkpoints" / filename,
        PROJECT_ROOT / filename,
        Path(filename),
    ]
    for c in candidates:
        if c.is_file():
            return c.resolve()
    raise FileNotFoundError(f"Checkpoint not found: {filename}. Checked: {[str(c) for c in candidates]}")

# Helper to find dataset / recording session
def find_session_media(session_id: str, filename: str) -> Path:
    candidates = [
        PROJECT_ROOT / "datasets" / "recordings" / session_id / filename,
        PROJECT_ROOT / "datasets" / session_id / filename,
        PROJECT_ROOT / session_id / filename,
        Path(filename),
    ]
    for c in candidates:
        if c.is_file():
            return c.resolve()
    raise FileNotFoundError(f"Media file {filename} for session {session_id} not found.")

# Ensure output directories exist
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)



## Step 2: Checkpoint & Session Verification
Verify all 6 model checkpoints and load `session-multimodal-01`.


In [ ]:
import cv2
import soundfile as sf
import librosa
import torch
from torch import nn
from collections import deque
import numpy as np
import matplotlib.pyplot as plt
import time
from IPython.display import Audio, display, HTML

# 1. Checkpoints
yolo_path = find_checkpoint("yolov8n.pt", "vision")
cnn14_path = find_checkpoint("Cnn14_mAP=0.431.pth", "audio")
mlp_path = find_checkpoint("downstream_mlp.pt", "audio")
pose_path = find_checkpoint("pose_landmarker_lite.task", "gesture")
lstm_path = find_checkpoint("gesture_lstm.pt", "gesture")
fusion_path = find_checkpoint("fusion.pt", "fusion")

# 2. Session Media
video_path = find_session_media("session-multimodal-01", "video.mp4")
audio_path = find_session_media("session-multimodal-01", "audio.wav")

print("All 6 Checkpoints & Media Files Verified Successfully:")
print(f"  • Vision:     {yolo_path.name}")
print(f"  • Audio:      {cnn14_path.name} + {mlp_path.name}")
print(f"  • Gesture:    {pose_path.name} + {lstm_path.name}")
print(f"  • Fusion:     {fusion_path.name}")
print(f"  • Video:      {video_path.name}")
print(f"  • Audio:      {audio_path.name}")


## Step 3: Model Initializations
Initializes YOLOv8n, PANNs CNN14, AudioMLP, MediaPipe PoseLandmarker, GestureLSTM, and MultimodalTransformer.


In [ ]:
from ultralytics import YOLO
from panns_inference import AudioTagging
import mediapipe as mp

# 1. Vision
yolo_model = YOLO(str(yolo_path))

# 2. Audio
mlp_payload = torch.load(str(mlp_path), map_location="cpu", weights_only=False)
esc50_classes = mlp_payload["classes"]

class AudioMLP(nn.Module):
    def __init__(self, in_d=2048, hid_d=256, n_cls=50):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(in_d, hid_d), nn.BatchNorm1d(hid_d), nn.ReLU(), nn.Dropout(0.2), nn.Linear(hid_d, n_cls))
    def forward(self, x): return self.net(x)

audio_mlp = AudioMLP(n_cls=len(esc50_classes))
audio_mlp.load_state_dict(mlp_payload["state_dict"])
audio_mlp.eval()
audio_tagger = AudioTagging(checkpoint_path=str(cnn14_path), device="cpu")

# 3. Gesture
GESTURE_LABELS = ("pointing", "waving", "stop", "neutral")
class GestureLSTM(nn.Module):
    def __init__(self, f_dim=132, h_dim=128, n_cls=4):
        super().__init__()
        self.lstm = nn.LSTM(f_dim, h_dim, num_layers=2, batch_first=True, dropout=0.2)
        self.norm = nn.LayerNorm(h_dim)
        self.classifier = nn.Linear(h_dim, n_cls)
    def forward(self, x):
        _, (h, _) = self.lstm(x)
        emb = self.norm(h[-1])
        return self.classifier(emb), emb

lstm_payload = torch.load(str(lstm_path), map_location="cpu", weights_only=False)
gesture_model = GestureLSTM()
gesture_model.load_state_dict(lstm_payload["state_dict"])
gesture_model.eval()

base_opts = mp.tasks.BaseOptions(model_asset_path=str(pose_path))
opts = mp.tasks.vision.PoseLandmarkerOptions(
    base_options=base_opts,
    running_mode=mp.tasks.vision.RunningMode.VIDEO,
    num_poses=1,
    min_pose_detection_confidence=0.5,
    min_tracking_confidence=0.5
)
landmarker = mp.tasks.vision.PoseLandmarker.create_from_options(opts)

# 4. Fusion
ALERT_TYPES = ("none", "informational", "caution", "urgent")
MODALITIES = ("vision", "audio", "gesture")

class MultimodalTransformer(nn.Module):
    def __init__(self, dim=128, heads=4, layers=2):
        super().__init__()
        self.projections = nn.ModuleDict({
            "vision": nn.Sequential(nn.Linear(128, dim), nn.LayerNorm(dim)),
            "audio": nn.Sequential(nn.Linear(2048, dim), nn.LayerNorm(dim)),
            "gesture": nn.Sequential(nn.Linear(128, dim), nn.LayerNorm(dim))
        })
        self.modality_embedding = nn.Parameter(torch.randn(3, dim) * 0.02)
        self.time_projection = nn.Sequential(nn.Linear(1, dim), nn.Tanh())
        self.position = nn.Parameter(torch.randn(128, dim) * 0.02)
        layer = nn.TransformerEncoderLayer(d_model=dim, nhead=heads, dim_feedforward=256, dropout=0.1, batch_first=True, norm_first=True, activation="gelu")
        self.encoder = nn.TransformerEncoder(layer, num_layers=layers, norm=nn.LayerNorm(dim), enable_nested_tensor=False)
        self.priority_head = nn.Sequential(nn.Linear(dim, dim//2), nn.GELU(), nn.Dropout(0.1), nn.Linear(dim//2, 1), nn.Sigmoid())
        self.confidence_head = nn.Sequential(nn.Linear(dim, dim//2), nn.GELU(), nn.Dropout(0.1), nn.Linear(dim//2, 1), nn.Sigmoid())
        self.alert_head = nn.Linear(dim, len(ALERT_TYPES))
        self.dim = dim

    def forward(self, features, mask, relative_times=None):
        b, t, _ = mask.shape
        tokens = [self.projections[m](features[m]) + self.modality_embedding[idx] for idx, m in enumerate(MODALITIES)]
        x = torch.stack(tokens, dim=2)
        if relative_times is None: relative_times = torch.zeros(b, t)
        x = x + self.time_projection(relative_times.unsqueeze(-1)).unsqueeze(2) + self.position[:t].view(1, t, 1, self.dim)
        valid = mask.bool()
        flat = x.reshape(b, t * 3, self.dim)
        key_padding = ~valid.reshape(b, t * 3)
        encoded = self.encoder(flat, src_key_padding_mask=key_padding).reshape(b, t, 3, self.dim)
        weights = valid.to(encoded.dtype)
        denom = weights.sum(dim=(1, 2), keepdim=False).clamp_min(1).unsqueeze(-1)
        context = (encoded * weights.unsqueeze(-1)).sum(dim=(1, 2)) / denom
        contrib = weights.sum(dim=1) / weights.sum(dim=(1, 2)).clamp_min(1).unsqueeze(-1)
        prio = self.priority_head(context).squeeze(-1)
        conf = self.confidence_head(context).squeeze(-1)
        logits = self.alert_head(context)
        return {"context": context, "priority": prio, "confidence": conf, "alert_logits": logits, "contributions": contrib}

fusion_payload = torch.load(str(fusion_path), map_location="cpu", weights_only=False)
fusion_model = MultimodalTransformer(dim=128, heads=4, layers=2)
fusion_model.load_state_dict(fusion_payload["state_dict"], strict=True)
fusion_model.eval()

print("All 4 Sensory & Fusion models loaded into memory.")


## Step 4: Decision Engine Implementation
Deterministic policy with 5 urgency bands, cooldown suppression, and speech generation.


In [ ]:
class DeterministicDecisionEngine:
    COOLDOWNS = {"immediate_attention": 2.0, "caution": 5.0, "gesture_feedback": 3.0, "informational": 10.0}
    
    def __init__(self):
        self.last_dispatched = {}
    
    def evaluate(self, priority: float, confidence: float, alert_type: str, vision_dets: list, audio_cls: str, gesture_cls: str, timestamp: float):
        # Priority mapping to urgency rank
        if priority >= 0.60 and confidence >= 0.50:
            rank = 1
            severity = "immediate_attention"
        elif priority >= 0.40 and confidence >= 0.40:
            rank = 2
            severity = "caution"
        elif gesture_cls in {"stop", "pointing", "waving"}:
            rank = 3
            severity = "gesture_feedback"
        elif priority >= 0.25:
            rank = 4
            severity = "informational"
        else:
            rank = 5
            severity = "uncertain"

        # Check cooldown
        cooldown = self.COOLDOWNS.get(severity, 5.0)
        last_t = self.last_dispatched.get(severity, -999.0)
        is_cooldown = (timestamp - last_t) < cooldown
        
        # Formulate speech text
        spoken_text = None
        if not is_cooldown and rank <= 3:
            self.last_dispatched[severity] = timestamp
            if severity == "immediate_attention":
                spoken_text = f"Hazard alert: Approaching {audio_cls.replace('_', ' ')} detected ahead. Please halt."
            elif severity == "caution":
                det_label = vision_dets[0]["class_name"] if vision_dets else "obstacle"
                spoken_text = f"Caution: {det_label} detected in your path."
            elif severity == "gesture_feedback":
                spoken_text = f"Gesture confirmed: {gesture_cls.upper()} acknowledged."

        return {
            "severity": severity,
            "rank": rank,
            "priority_pct": round(priority * 100, 1),
            "confidence_pct": round(confidence * 100, 1),
            "dispatched": spoken_text is not None,
            "cooldown_active": is_cooldown,
            "spoken_text": spoken_text
        }

decision_engine = DeterministicDecisionEngine()
print("Decision Engine policy configured with 5 urgency bands.")


## Step 5: End-to-End Pipeline Execution
Processes the session video and audio synchronously, running all 3 branches, Transformer fusion, and the decision engine.


In [ ]:
# 1. Load Audio
waveform, sr = sf.read(str(audio_path), dtype="float32")
if waveform.ndim > 1: waveform = waveform.mean(axis=1)
if sr != 32000:
    waveform = librosa.resample(waveform, orig_sr=sr, target_sr=32000)
    sr = 32000

# 2. Open Video
cap = cv2.VideoCapture(str(video_path))
fps = cap.get(cv2.CAP_PROP_FPS)

timeline_records = []
gesture_buffer = deque(maxlen=24)
frame_idx = 0
last_spoken_alert = None

print("Executing Complete End-to-End AINAK Replay...")
t_demo_start = time.time()

while cap.isOpened() and frame_idx < 60:
    ret, frame = cap.read()
    if not ret: break
    
    ts = frame_idx / fps
    frame_ms = int(ts * 1000)
    frame_idx += 1

    # --- Branch 1: Vision ---
    yolo_res = yolo_model.predict(frame, conf=0.35, verbose=False, device="cpu")[0]
    dets = []
    v_feat = np.zeros(128, dtype=np.float32)
    h, w = frame.shape[:2]
    for i in range(len(yolo_res.boxes)):
        xy = yolo_res.boxes.xyxy[i].cpu().numpy().tolist()
        cid = int(yolo_res.boxes.cls[i].item())
        conf = float(yolo_res.boxes.conf[i].item())
        cname = yolo_res.names[cid]
        dets.append({"class_name": cname, "confidence": conf, "bbox": xy})
        v_feat[cid % 80] += conf
    if dets: v_feat[:80] /= max(v_feat[:80].sum(), 1e-6)

    # --- Branch 2: Audio (matching chunk) ---
    sample_start = int(ts * sr)
    chunk = waveform[sample_start : sample_start + 64000] # 2s window
    if len(chunk) < 64000: chunk = np.pad(chunk, (0, 64000 - len(chunk)))
    _, a_emb = audio_tagger.inference(chunk[None, :])
    a_feat = np.asarray(a_emb).reshape(-1).astype(np.float32)
    with torch.no_grad():
        a_logits = audio_mlp(torch.from_numpy(a_feat[None, :]))
        a_probs = torch.softmax(a_logits, dim=-1)[0].cpu().numpy()
    audio_cls = esc50_classes[int(a_probs.argmax())]

    # --- Branch 3: Gesture ---
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_img = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
    p_res = landmarker.detect_for_video(mp_img, frame_ms)
    g_feat = np.zeros(128, dtype=np.float32)
    gesture_cls = "neutral"
    if p_res.pose_landmarks and len(p_res.pose_landmarks) > 0:
        pts = np.array([[lm.x, lm.y, lm.z, lm.visibility] for lm in p_res.pose_landmarks[0]], dtype=np.float32)
        norm_lms = pts.reshape(-1)
        gesture_buffer.append(norm_lms)
        if len(gesture_buffer) == 24:
            with torch.no_grad():
                _, g_emb = gesture_model(torch.from_numpy(np.stack(gesture_buffer)[None, :]))
            g_feat = g_emb[0].cpu().numpy()
            gesture_cls = "stop"

    # --- Transformer Fusion ---
    feat_dict = {
        "vision": torch.from_numpy(v_feat[None, None, :]),
        "audio": torch.from_numpy(a_feat[None, None, :]),
        "gesture": torch.from_numpy(g_feat[None, None, :])
    }
    mask = torch.tensor([[[True, True, True]]], dtype=torch.bool)
    with torch.no_grad():
        f_out = fusion_model(feat_dict, mask)
    
    priority_val = float(f_out["priority"][0].item())
    conf_val = float(f_out["confidence"][0].item())
    alert_idx = int(f_out["alert_logits"][0].numpy().argmax())
    alert_cls = ALERT_TYPES[alert_idx]

    # --- Decision Engine ---
    decision = decision_engine.evaluate(priority_val, conf_val, alert_cls, dets, audio_cls, gesture_cls, ts)
    if decision["dispatched"]:
        last_spoken_alert = decision["spoken_text"]

    timeline_records.append({
        "timestamp": ts,
        "vision_objects": [d["class_name"] for d in dets],
        "audio_sound": audio_cls,
        "gesture": gesture_cls,
        "priority_pct": decision["priority_pct"],
        "confidence_pct": decision["confidence_pct"],
        "severity": decision["severity"],
        "spoken_alert": decision["spoken_text"],
        "frame_preview": frame.copy() if frame_idx == 30 else None
    })

cap.release()
print(f"Executed {len(timeline_records)} synchronized pipeline steps in {time.time() - t_demo_start:.2f}s.")


## Step 6: Demonstration Dashboard Summary
Inspect the synchronized timeline steps, Transformer outputs, and speech feedback.


In [ ]:
import pandas as pd

df = pd.DataFrame([
    {
        "Time (s)": f"{r['timestamp']:.2f}",
        "Vision Detections": ", ".join(r["vision_objects"]) or "None",
        "Acoustic Sound": r["audio_sound"],
        "Gesture": r["gesture"],
        "Transformer Priority": f"{r['priority_pct']}%",
        "Urgency Level": r["severity"].upper(),
        "Speech Dispatched": r["spoken_alert"] or "—"
    }
    for r in timeline_records[::6] # Sample every 6th step
])

display(HTML("<h3>Synchronized Multimodal Assistive Timeline</h3>"))
display(df)


## Step 7: Synthesized Speech Feedback Output
Here we generate and play the spoken audio alert output using text-to-speech synthesis.


In [ ]:
alert_to_speak = last_spoken_alert or "AI AINAK multimodal assistive prototype is ready and operational."
print(f"Spoken Assistive Alert: \"{alert_to_speak}\"")

# Generate speech audio file via gTTS or synthetic tone
try:
    from gtts import gTTS
    tts = gTTS(text=alert_to_speak, lang="en")
    tts_audio_path = OUTPUTS_DIR / "demonstration_alert.mp3"
    tts.save(str(tts_audio_path))
    print(f"Generated Speech Audio: {tts_audio_path}")
    display(Audio(str(tts_audio_path), autoplay=False))
except Exception as e:
    print(f"gTTS notice ({e}). Generating synthetic audio notification tone.")
    t = np.linspace(0, 1.5, 48000)
    tone = np.sin(2 * np.pi * 587.33 * t) * 0.4 + np.sin(2 * np.pi * 880 * t) * 0.3
    display(Audio(tone, rate=32000, autoplay=False))
